# 03 Ablation Testing

**Author:** Rowan Walker

This notebook captures ablation testing. Elements of the transformer architecture have been removed to determine the efficacy of these features and the robustness of the model design choices.

### 3.1 Imports

In [17]:
import numpy as np
import pandas as pd

from src.data.preprocess import generate_valid_tickers, generate_model_inputs
from src.inference.inference import model_inference
from src.training.train import train_model
from src.utils.backtest import backtest, optimise_sharpe
from src.utils.risk import full_risk_report
from src.utils.seed import set_global_seed

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch")
warnings.filterwarnings("ignore", category=FutureWarning, module="torch")
warnings.filterwarnings("ignore", category=UserWarning, module="hmmlearn")

%reload_ext autoreload
%autoreload 2

# Global seed setting for reproducibility
set_global_seed()

#### Setting a common parameter grid for the back test

In [2]:
param_grid = {
    'long_threshold': 0.6, 
    'short_threshold': 0.4, 
    'target_vol': 0.2,  
    'slippage': 1.0,
    'commission': 1.0,
    'take_profit': 0.16,
    'stop_loss': -0.02,
    'max_hold_days': 18.0,
    'max_drawdown': 0.2,
    'leverage': 2.0}

#### Defining an E2E model run

In [3]:
def e2e(param_grid, ablation_name):
    for i, t in enumerate((1, 5, 21)):
        valid_tickers = generate_valid_tickers(
            start_date='2010-12-01',
            end_date='2025-12-01')
        
        X, y, stock_ids, regime_X, full_df = generate_model_inputs(
            tickers=valid_tickers,
            train_start='2010-12-01',
            train_end='2019-12-01',
            hold_days=t,
            verbose=False
        )

        train_model(
            X, 
            y, 
            stock_ids, 
            regime_X, 
            hold_days=t,
            ablation_name=ablation_name,
            verbose=False
        )

    model_inference(
        valid_tickers, 
        full_df, 
        feature_dim=X.shape[2], 
        hold_days=(1,5,21),
        verbose=False
    )
    
    sharpe = backtest(param_grid, start_date='2020-01-01', end_date='2025-12-31', sharpe_only=True)

    return sharpe

## 3.2 Ablations

### 3.2.1 Test 1: Core model run

**Ablation config grid (default):** </br>
&emsp;use_regime_embedding: _true_</br>
&emsp;shuffle_regime: _false_</br>
&emsp;constant_regime: _false_</br>
&emsp;use_stock_embedding: _true_</br>
&emsp;use_cls_token: _true_

### 3.2.2 Test 2: Removing regime embedding

**Ablation config grid:** </br>
&emsp;use_regime_embedding: _**false**_</br>
&emsp;shuffle_regime: _false_</br>
&emsp;constant_regime: _false_</br>
&emsp;use_stock_embedding: _true_</br>
&emsp;use_cls_token: _true_

### 3.2.2 Test 3: Shuffle regime

**Ablation config grid:** </br>
&emsp;use_regime_embedding: _true_</br>
&emsp;shuffle_regime: _**true**_</br>
&emsp;constant_regime: _false_</br>
&emsp;use_stock_embedding: _true_</br>
&emsp;use_cls_token: _true_

### 3.2.2 Test 4: Constant regime

**Ablation config grid:** </br>
&emsp;use_regime_embedding: _true_</br>
&emsp;shuffle_regime: _false_</br>
&emsp;constant_regime: _**true**_</br>
&emsp;use_stock_embedding: _true_</br>
&emsp;use_cls_token: _true_

### 3.2.2 Test 5: Remove stock embedding

**Ablation config grid:** </br>
&emsp;use_regime_embedding: _true_</br>
&emsp;shuffle_regime: _false_</br>
&emsp;constant_regime: _false_</br>
&emsp;use_stock_embedding: _**false**_</br>
&emsp;use_cls_token: _true_

### 3.2.2 Test 6: Remove CLS token

**Ablation config grid:** </br>
&emsp;use_regime_embedding: _true_</br>
&emsp;shuffle_regime: _false_</br>
&emsp;constant_regime: _false_</br>
&emsp;use_stock_embedding: _true_</br>
&emsp;use_cls_token: _**false**_

## 3.3 Results

In [30]:
ablation_names = (
    'baseline',
    'no_regime',
    'shuffle_regime',
    'constant_regime',
    'no_stock_embedding',
    'no_cls_token'
)

results = {}

for name in ablation_names:
    sharpe = e2e(param_grid, name)
    results[name] = round(sharpe, 2)

{'baseline': 3.23,
 'no_regime': 2.85,
 'shuffle_regime': 3.47,
 'constant_regime': 3.57,
 'no_stock_embedding': 3.14,
 'no_cls_token': 3.82}

In [49]:
pd.DataFrame.from_dict(results, orient='index', columns=['Sharpe'])

,Sharpe
baseline,3.23
no_regime,2.85
shuffle_regime,3.47
constant_regime,3.57
no_stock_embedding,3.14
no_cls_token,3.82


## 3.5 Conclusion

This ablation study provides several clear insights into the model’s architecture and information usage. The baseline model exhibits strong and stable performance, indicating that the core transformer-based time-series representation is robust. Removing regime information leads to a modest deterioration in performance, suggesting that regime features contribute incremental value but are not essential to the model’s predictive power.

Replacing true regime inputs with shuffled or constant regimes improves performance relative to the baseline. This indicates that regime awareness may act as a useful structural or regularising signal, the specific regime estimates themselves likely introduce noise. This is likely due to instability and uncertainty of unsupervised regime detection methods.

Removing stock embeddings has little effect on performance, implying that cross-sectional differentiation is largely captured through the input features themselves. In contrast, removing the CLS token leads to a significant performance improvement, suggesting that explicit sequence pooling is better suited to this task than CLS-based aggregation.

Overall, the results indicate that the model is not reliant on any single architectural component and generalises well across ablations. The findings motivate a simplified final architecture that removes the CLS token and treats regime information as a weak or regularising signal rather than a primary source of predictive information.